# Tangling and effective dimensionality — discrete-trained checkpoints

Same analysis as `10_tangling_dim_evolution.ipynb` but loads the
discrete-training checkpoints `stage{0,1,2,3}_topo_discrete.pt`
produced by `01d_train_topo_discrete.ipynb`.

Hypothesis: with discrete 4-quadrant training, conditions inside each
CTOA bin are 4 well-separated spatial clusters (instead of uniformly
distributed angles), so trial-averaging in `prepare_jpca_input` keeps
spatial structure → PR opens above 1 → inverted-U Q vs CTOA, as in
Amengual et al.


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials, filter_trials,
    tangling_by_ctoa_bin, prepare_jpca_input,
    participation_ratio,
)

device = "cpu"

def make_model():
    return BioLeakyRNNTopo(
        input_size=7, hidden_size=180, output_size=2,
        dt=20.0, tau=100.0, activation="softplus",
        sigma_rec=0.10, rec_init="diag",
        use_ei=True, exc_ratio=0.80, use_dale=True,
        mask_seed=42, sheet_side=12,
        tau_ee=0.25, tau_ie=0.32, tau_ei=0.64, tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=True,
        tau_adapt=100.0,
        g_adapt=0.5,
    ).to(device)

def make_env():
    # IMPORTANT: eval uses discrete (4-quadrant) target locations even
    # though training was continuous. With continuous targets,
    # prepare_jpca_input averages h(t) across uniformly-distributed
    # angles within a CTOA bin -> spatial signal cancels by rotational
    # averaging, the inverted-U structure in tangling/dim collapses.
    # 4 discrete quadrants give clean spatial conditions per CTOA bin
    # (this is also what Amengual recorded in monkey).
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0,
        p_distractor_trial=0.6, distractor_strength=1.0,
        continuous_locations=False,
    )

from src.figsave import autosave
autosave("11_tangling_dim_discrete")


In [ ]:
def pr_paperstyle_by_ctoa_bin(trials, align_key="target_onset",
                              window_before=15, window_after=0,
                              outcome="correct", n_components=3):
    """PR per CTOA bin, restricted to the top-`n_components` PCs of the
    population activity. PCA is fit on the concatenated bins so the
    3-D subspace is shared and per-bin PR reflects how much of that
    subspace is excited in this regime."""
    X_cond, labels, rel_time, counts = prepare_jpca_input(
        trials, align_key=align_key,
        window_before=window_before, window_after=window_after,
        min_trials=5, outcome=outcome,
    )
    if X_cond is None:
        return None
    C, T, N = X_cond.shape
    X_flat = X_cond.reshape(C * T, N)
    k = min(n_components, N, X_flat.shape[0])
    pca = PCA(n_components=k).fit(X_flat)
    PR = []
    for c in range(C):
        Z = pca.transform(X_cond[c])
        PR.append(participation_ratio(Z))
    return {
        "PR":           np.array(PR),
        "labels":       labels,
        "rel_time":     rel_time,
        "counts":       counts,
        "n_components": k,
        "explained":    float(pca.explained_variance_ratio_.sum()),
    }

def ctoa_ms_mean(trials, labels, outcome="correct"):
    bin_to_ms = defaultdict(list)
    for tr in trials:
        if tr.get("train_outcome") != outcome:
            continue
        b, ms = tr.get("ctoa_bin"), tr.get("ctoa_ms")
        if b in labels and ms is not None:
            bin_to_ms[b].append(ms)
    return np.array([np.mean(bin_to_ms[b]) if b in bin_to_ms else np.nan
                      for b in labels])


In [ ]:
STAGES = ["stage0_topo_discrete", "stage1_topo_discrete", "stage2_topo_discrete", "stage3_topo_discrete"]
N_TRIALS = 2000   # match 04b for solid per-bin statistics
WINDOW_BEFORE = 15      # 300 ms pre-target at dt=20
WINDOW_AFTER  = 0
PCA_DIMS_FOR_Q = 20

results = {}
for name in STAGES:
    ckpt_path = Path(f"checkpoints/{name}.pt")
    if not ckpt_path.exists():
        print(f"[skip] {name}: missing {ckpt_path}"); continue
    print(f"[{name}] loading + collecting {N_TRIALS} trials...", flush=True)
    m = make_model()
    state = torch.load(ckpt_path, weights_only=True)["state_dict"]
    m.load_state_dict(state, strict=False); m.eval()
    m.noise_at_eval = True   # CRITICAL: without this every trial is identical
    torch.manual_seed(0); np.random.seed(0)
    trials = collect_trials(m, make_env, n_trials=N_TRIALS, device=device)
    corr   = filter_trials(trials, outcome="correct")
    print(f"  correct {len(corr)}/{len(trials)}")
    if len(corr) < 30:
        print("  too few correct trials"); continue
    tang = tangling_by_ctoa_bin(
        corr, window_before=WINDOW_BEFORE, window_after=WINDOW_AFTER,
        pca_dims=PCA_DIMS_FOR_Q, outcome="correct",
    )
    pr3 = pr_paperstyle_by_ctoa_bin(
        corr, window_before=WINDOW_BEFORE, window_after=WINDOW_AFTER,
        n_components=3,
    )
    if tang is None or pr3 is None:
        print("  no per-bin data"); continue
    pr3["ctoa_ms_mean"] = ctoa_ms_mean(corr, pr3["labels"])
    results[name] = {
        "tang": tang, "pr3": pr3,
        "n_correct": len(corr), "n_total": len(trials),
    }
    print(f"  PR-3 explained variance: {pr3['explained']:.3f}")
print("done.")


In [ ]:
def quad_fit(x, y):
    """Returns (coeffs, x_smooth, y_smooth, R2). NaN-safe."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return None
    p = np.polyfit(x[m], y[m], 2)
    yhat = np.polyval(p, x[m])
    ss_res = np.sum((y[m] - yhat) ** 2)
    ss_tot = np.sum((y[m] - y[m].mean()) ** 2)
    R2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    xs = np.linspace(x[m].min(), x[m].max(), 200)
    return p, xs, np.polyval(p, xs), R2


In [ ]:
stage_color = {"stage0_topo_discrete": "#888888", "stage1_topo_discrete": "#5b9bd5",
               "stage2_topo_discrete": "#ed7d31", "stage3_topo_discrete": "#c00000"}

fig, axes = plt.subplots(1, len(results), figsize=(4 * max(len(results), 1), 4),
                          sharey=True)
if len(results) == 1: axes = [axes]
for ax, (name, R) in zip(axes, results.items()):
    pr = R["pr3"]
    x = pr["ctoa_ms_mean"]; y = pr["PR"]
    ax.scatter(x, y, s=55, color=stage_color.get(name, "k"), zorder=3,
               edgecolor="k", linewidth=0.5)
    fit = quad_fit(x, y)
    if fit is not None:
        _, xs, ys, R2 = fit
        ax.plot(xs, ys, "--", color="red", lw=1.5, zorder=2)
        ax.text(0.62, 0.85, f"R² = {R2:.2f}", transform=ax.transAxes,
                color="red", fontsize=11)
    ax.set_title(f"{name}\nn_corr={R['n_correct']}", fontsize=10)
    ax.set_xlabel("CTOA (ms)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Dimensionality (PR on 3 PCs)")
fig.suptitle("Effective dimensionality vs CTOA — Amengual Fig. 5 style", y=1.02)
fig.tight_layout()
fig.savefig("figures/dim_evolution_discrete.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4 * max(len(results), 1), 4),
                          sharey=True)
if len(results) == 1: axes = [axes]
for ax, (name, R) in zip(axes, results.items()):
    t = R["tang"]
    x = t["ctoa_ms_mean"]; y = t["Q_mean"]
    ax.scatter(x, y, s=55, color=stage_color.get(name, "k"), zorder=3,
               edgecolor="k", linewidth=0.5)
    fit = quad_fit(x, y)
    if fit is not None:
        _, xs, ys, R2 = fit
        ax.plot(xs, ys, "--", color="red", lw=1.5, zorder=2)
        ax.text(0.62, 0.85, f"R² = {R2:.2f}", transform=ax.transAxes,
                color="red", fontsize=11)
    ax.set_title(f"{name}\nn_corr={R['n_correct']}", fontsize=10)
    ax.set_xlabel("CTOA (ms)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Tangling Q (pre-target)")
fig.suptitle("Pre-target tangling vs CTOA — Amengual Fig. 7 style", y=1.02)
fig.tight_layout()
fig.savefig("figures/tangling_evolution_discrete.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
print(f"{'stage':<14} {'n_corr':>7} {'PR R²':>8} {'PR coef[a,b,c]':>34}  {'Q  R²':>8}")
for name, R in results.items():
    pr = R["pr3"]; t = R["tang"]
    pr_fit = quad_fit(pr["ctoa_ms_mean"], pr["PR"])
    q_fit  = quad_fit(t["ctoa_ms_mean"], t["Q_mean"])
    pr_R2 = pr_fit[3] if pr_fit else float("nan")
    q_R2  = q_fit[3]  if q_fit  else float("nan")
    pr_p  = pr_fit[0] if pr_fit else [np.nan]*3
    print(f"{name:<14} {R['n_correct']:>7} {pr_R2:>8.3f} "
          f"[{pr_p[0]:+.2e},{pr_p[1]:+.2e},{pr_p[2]:+.2e}]  {q_R2:>8.3f}")
print()
print("Inverted-U iff PR coef[0] (a) < 0 and R² above ~0.4.")


## Reading the results

**Inverted-U appears** ⇔ leading quadratic coefficient `a < 0` AND R² ≳ 0.4.
If `a > 0` (U opening upward), tangling/dim are *higher* at the
extremes — that's a monotonic-fatigue regime, not the paper's hazard-rate
structure.

Stage progression: stage0 should be near-flat (no cue, no temporal
structure used); curves should sharpen across stage1 → stage2 → stage3 as
the network adds first cue use, then distractor robustness, then
preparatory structure.
